## Streamlit Dashboard

**Goal:** Build the full 4-section merchant dashboard in Streamlit.

**Sections:**
1. Metrics Banner (revenue, net cash flow, WoW change, avg txn)
2. Revenue Trend + 7-Day Forecast Chart
3. Credit Readiness Gauge + Signal Breakdown
4. AI Advisor Chat Card

## 1. Test Dashboard Data Pipeline

Verify all modules from days 3-5 are working before wiring into Streamlit.

In [ ]:
import os, sys, json
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
import plotly.graph_objects as go
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text

load_dotenv()
sys.path.insert(0, ".")
engine = create_engine(os.getenv("DATABASE_URL"))

# Import all modules
from src.analytics.cashflow import get_analytics
from src.analytics.credit_score import compute_credit_score
from src.advice.advisor import generate_ai_advice
from openai import OpenAI


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MERCHANT_ID = "DEMO_001"

analytics = get_analytics(MERCHANT_ID, engine)
credit    = compute_credit_score(MERCHANT_ID, engine)
advice    = generate_ai_advice(MERCHANT_ID, analytics, credit, openai_client=openai_client)

print(" All modules loaded successfully.")
print(f"   Revenue:      ₦{analytics['total_revenue_ngn']:,.2f}")
print(f"   Credit Score: {credit['score']}/850 — {credit['band']}")
print(f"   AI Advice:    {advice['advice'][:70]}...")

In [ ]:
# Chart 1: Revenue Trend + Forecast
def build_revenue_forecast_chart(analytics: dict) -> go.Figure:
    daily = analytics.get("daily_revenue", [])
    forecast_data = analytics.get("forecast_7day", {}).get("forecast", [])

    hist_dates = [str(d["date"]) for d in daily]
    hist_rev   = [d["revenue"]   for d in daily]
    fore_dates = [f["date"]             for f in forecast_data]
    fore_rev   = [f["predicted_revenue"] for f in forecast_data]
    fore_upper = [f.get("upper_bound", f["predicted_revenue"] * 1.2) for f in forecast_data]
    fore_lower = [f.get("lower_bound", f["predicted_revenue"] * 0.8) for f in forecast_data]

    fig = go.Figure()

    # Actual revenue bars
    fig.add_trace(go.Bar(
        x=hist_dates, y=hist_rev,
        name="Actual Revenue",
        marker_color="#22c55e", opacity=0.85
    ))

    # Forecast line
    fig.add_trace(go.Scatter(
        x=fore_dates, y=fore_rev,
        name="7-Day Forecast",
        mode="lines+markers",
        line=dict(color="#f59e0b", width=2, dash="dash"),
        marker=dict(size=7, symbol="diamond")
    ))

    # Confidence band
    fig.add_trace(go.Scatter(
        x=fore_dates + fore_dates[::-1],
        y=fore_upper + fore_lower[::-1],
        fill="toself",
        fillcolor="rgba(245,158,11,0.12)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Forecast Range", showlegend=True
    ))

    fig.update_layout(
        title="Revenue Trend & 7-Day Forecast",
        xaxis_title="Date",
        yaxis_title="Revenue (₦)",
        template="plotly_white",
        height=380,
        legend=dict(orientation="h", y=-0.2),
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig


# Chart 2: Credit Score Gauge
def build_credit_gauge(credit: dict) -> go.Figure:
    score = credit.get("score", 0)
    band  = credit.get("band", "N/A")
    color = credit.get("band_color", "#6b7280")

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        title={"text": f"Credit Readiness — <b>{band}</b>", "font": {"size": 14}},
        number={"suffix": " / 850", "font": {"size": 28}},
        gauge={
            "axis": {"range": [0, 850], "tickwidth": 1, "tickfont": {"size": 10}},
            "bar": {"color": color, "thickness": 0.3},
            "bgcolor": "white",
            "borderwidth": 2,
            "steps": [
                {"range": [0,   449], "color": "#fecaca"},
                {"range": [449, 549], "color": "#fed7aa"},
                {"range": [549, 649], "color": "#fef08a"},
                {"range": [649, 749], "color": "#bbf7d0"},
                {"range": [749, 850], "color": "#86efac"},
            ],
        }
    ))
    fig.update_layout(height=280, margin=dict(l=10, r=10, t=40, b=10))
    return fig


# Chart 3: Signal Bar Chart
def build_signal_bars(credit: dict) -> go.Figure:
    signals = credit.get("signals", [])
    if not signals:
        return go.Figure()

    labels = [s["signal"].replace("_", " ").title() for s in signals]
    pcts   = [round(s["score"] / s["max"] * 100) for s in signals]
    colors = ["#16a34a" if p >= 70 else "#ca8a04" if p >= 40 else "#dc2626" for p in pcts]

    fig = go.Figure(go.Bar(
        x=pcts, y=labels,
        orientation="h",
        marker_color=colors,
        text=[f"{p}%" for p in pcts],
        textposition="outside"
    ))
    fig.update_layout(
        title="Credit Signal Breakdown",
        xaxis=dict(range=[0, 110], title="Score %"),
        template="plotly_white",
        height=260,
        margin=dict(l=10, r=30, t=40, b=10)
    )
    return fig


# Chart 4: Category Pie Chart
def build_category_pie(analytics: dict) -> go.Figure:
    cat_data = analytics.get("category_breakdown", {})
    if not cat_data:
        return go.Figure()

    labels = [k.replace("_", " ").title() for k in cat_data.keys()]
    values = list(cat_data.values())

    fig = go.Figure(go.Pie(
        labels=labels, values=values,
        hole=0.4,
        textinfo="label+percent",
        marker=dict(colors=px.colors.qualitative.Set3)
    ))
    fig.update_layout(
        title="Revenue by Category",
        height=300,
        margin=dict(l=10, r=10, t=40, b=10),
        showlegend=False
    )
    return fig


# Quick preview all charts
build_revenue_forecast_chart(analytics).show()
build_credit_gauge(credit).show()
build_signal_bars(credit).show()
build_category_pie(analytics).show()
print("All 4 charts rendered.")

## Write the Streamlit App (`dashboard/app.py`)

In [ ]:
os.makedirs("dashboard", exist_ok=True)

streamlit_app = '''
import os
import sys
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine
from openai import OpenAI

# ── Path setup ────────────────────────────────────────────────────────
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
load_dotenv()

from src.analytics.cashflow import get_analytics
from src.analytics.credit_score import compute_credit_score
from src.advice.advisor import generate_ai_advice

# ── Page config ───────────────────────────────────────────────────────
st.set_page_config(
    page_title="NombaIQ — Merchant Intelligence",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ────────────────────────────────────────────────────────
st.markdown("""
<style>
    .main-header {
        font-size: 2rem; font-weight: 800; color: #1a1a2e;
        border-bottom: 3px solid #16a34a; padding-bottom: 8px; margin-bottom: 24px;
    }
    .metric-card {
        background: linear-gradient(135deg, #f8fafc, #f1f5f9);
        border: 1px solid #e2e8f0; border-radius: 12px;
        padding: 16px 20px; text-align: center;
    }
    .metric-value { font-size: 1.6rem; font-weight: 700; color: #0f172a; }
    .metric-label { font-size: 0.78rem; color: #64748b; text-transform: uppercase; letter-spacing: 0.05em; }
    .metric-delta-pos { color: #16a34a; font-size: 0.85rem; }
    .metric-delta-neg { color: #dc2626; font-size: 0.85rem; }
    .advice-card {
        background: linear-gradient(135deg, #ecfdf5, #f0fdf4);
        border: 1px solid #86efac; border-radius: 12px;
        padding: 20px 24px; margin-top: 8px;
    }
    .advice-text { font-size: 1.05rem; line-height: 1.7; color: #14532d; }
    .section-header { font-size: 1.15rem; font-weight: 700; color: #0f172a; margin-bottom: 12px; }
    .credit-band-badge {
        display: inline-block; padding: 4px 12px; border-radius: 20px;
        font-weight: 700; font-size: 0.85rem;
    }
    div[data-testid="stMetric"] { background: #f8fafc; border-radius: 10px; padding: 12px; }
</style>
""", unsafe_allow_html=True)


# ── DB + OpenAI (cached) ──────────────────────────────────────────────
@st.cache_resource
def get_engine():
    return create_engine(os.getenv("DATABASE_URL"))

@st.cache_resource
def get_openai():
    return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


@st.cache_data(ttl=120)
def load_dashboard_data(merchant_id: str):
    eng = get_engine()
    analytics = get_analytics(merchant_id, eng)
    credit    = compute_credit_score(merchant_id, eng)
    return analytics, credit


# ── Sidebar ───────────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://nomba.com/favicon.ico", width=32)
    st.markdown("## NombaIQ")
    st.caption("Merchant Intelligence Platform")
    st.divider()

    merchant_id = st.text_input(
        "Merchant ID",
        value="DEMO_001",
        placeholder="e.g. DEMO_001",
        help="Enter your Nomba merchant ID"
    )

    st.divider()
    st.caption("💬 Ask your AI Advisor")
    user_question = st.text_area(
        "Your question",
        placeholder="e.g. How much loan can I get?\nWhere is my money going?",
        height=100, label_visibility="collapsed"
    )
    ask_btn = st.button("Ask NombaIQ", type="primary", use_container_width=True)

    st.divider()
    refresh_btn = st.button("🔄 Refresh Data", use_container_width=True)
    if refresh_btn:
        st.cache_data.clear()
        st.rerun()

    st.caption(f"Powered by Nomba Webhooks + Prophet + GPT-4o mini")


# ── Main Header ───────────────────────────────────────────────────────
st.markdown(\'<div class="main-header">📊 NombaIQ Merchant Dashboard</div>\', unsafe_allow_html=True)

if not merchant_id:
    st.info("Enter a Merchant ID in the sidebar to get started.")
    st.stop()

# ── Load data ─────────────────────────────────────────────────────────
with st.spinner("Loading merchant data..."):
    try:
        analytics, credit = load_dashboard_data(merchant_id)
    except Exception as e:
        st.error(f"Could not load data for {merchant_id}: {e}")
        st.stop()

if not analytics:
    st.warning(f"No transaction data found for merchant **{merchant_id}**. Run the webhook simulation in Day 1 notebook.")
    st.stop()


# ════════════════════════════════════════════════════════════════════
# SECTION 1 — METRICS BANNER
# ════════════════════════════════════════════════════════════════════
st.markdown(\'<div class="section-header">📈 Business Overview</div>\', unsafe_allow_html=True)

c1, c2, c3, c4, c5 = st.columns(5)

with c1:
    st.metric(
        "Total Revenue",
        f"₦{analytics[\'total_revenue_ngn\']/1000:.1f}K",
        help="All-time cumulative revenue"
    )

with c2:
    net = analytics[\'net_cash_flow_ngn\']
    st.metric(
        "Net Cash Flow",
        f"₦{net/1000:.1f}K",
        delta=f"{'Positive' if net >= 0 else 'Negative'}",
        delta_color="normal" if net >= 0 else "inverse"
    )

with c3:
    wow = analytics[\'wow_change_pct\']
    st.metric(
        "Week-on-Week",
        f"{wow:+.1f}%",
        delta=f"vs last week",
        delta_color="normal" if wow >= 0 else "inverse"
    )

with c4:
    st.metric(
        "Avg Transaction",
        f"₦{analytics[\'avg_transaction_ngn\']:,.0f}"
    )

with c5:
    st.metric(
        "Total Transactions",
        f"{analytics[\'transaction_count\']}"
    )

st.divider()


# ════════════════════════════════════════════════════════════════════
# SECTION 2 — REVENUE TREND + FORECAST
# ════════════════════════════════════════════════════════════════════
col_left, col_right = st.columns([3, 1])

with col_left:
    st.markdown(\'<div class="section-header">📅 Revenue Trend & 7-Day Forecast</div>\', unsafe_allow_html=True)

    daily = analytics.get("daily_revenue", [])
    forecast_data = analytics.get("forecast_7day", {}).get("forecast", [])

    hist_dates = [str(d["date"]) for d in daily]
    hist_rev   = [d["revenue"]   for d in daily]
    fore_dates = [f["date"] for f in forecast_data]
    fore_rev   = [f["predicted_revenue"] for f in forecast_data]
    fore_upper = [f.get("upper_bound", f["predicted_revenue"]*1.2) for f in forecast_data]
    fore_lower = [f.get("lower_bound", f["predicted_revenue"]*0.8) for f in forecast_data]

    fig_trend = go.Figure()
    fig_trend.add_trace(go.Bar(x=hist_dates, y=hist_rev, name="Actual Revenue", marker_color="#22c55e", opacity=0.85))
    fig_trend.add_trace(go.Scatter(x=fore_dates, y=fore_rev, name="Forecast", mode="lines+markers", line=dict(color="#f59e0b", width=2, dash="dash"), marker=dict(size=7)))
    if fore_upper and fore_lower:
        fig_trend.add_trace(go.Scatter(x=fore_dates+fore_dates[::-1], y=fore_upper+fore_lower[::-1], fill="toself", fillcolor="rgba(245,158,11,0.12)", line=dict(color="rgba(255,255,255,0)"), name="Forecast Range"))
    fig_trend.update_layout(template="plotly_white", height=360, legend=dict(orientation="h", y=-0.25), margin=dict(l=5,r=5,t=20,b=5), xaxis_title="Date", yaxis_title="Revenue (₦)")
    st.plotly_chart(fig_trend, use_container_width=True)

with col_right:
    st.markdown(\'<div class="section-header">📊 Category Mix</div>\', unsafe_allow_html=True)
    cat_data = analytics.get("category_breakdown", {})
    if cat_data:
        labels = [k.replace("_"," ").title() for k in cat_data.keys()]
        values = list(cat_data.values())
        fig_pie = go.Figure(go.Pie(labels=labels, values=values, hole=0.45, textinfo="percent", marker=dict(colors=px.colors.qualitative.Set3)))
        fig_pie.update_layout(height=360, margin=dict(l=5,r=5,t=5,b=5), showlegend=True, legend=dict(font=dict(size=10)))
        st.plotly_chart(fig_pie, use_container_width=True)

    # 7-day forecast summary table
    st.caption("7-Day Forecast")
    if forecast_data:
        import pandas as pd
        df_fc = pd.DataFrame(forecast_data)[["date","predicted_revenue"]]
        df_fc["predicted_revenue"] = df_fc["predicted_revenue"].apply(lambda x: f"₦{x:,.0f}")
        df_fc.columns = ["Date", "Forecast"]
        st.dataframe(df_fc, hide_index=True, use_container_width=True)

st.divider()


# ════════════════════════════════════════════════════════════════════
# SECTION 3 — CREDIT READINESS SCORE
# ════════════════════════════════════════════════════════════════════
st.markdown(\'<div class="section-header">🏦 Credit Readiness Score</div>\', unsafe_allow_html=True)

col_gauge, col_signals, col_reco = st.columns([1.2, 1.5, 1.3])

with col_gauge:
    score = credit.get("score", 0)
    band  = credit.get("band", "N/A")
    color = credit.get("band_color", "#6b7280")
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        title={"text": f"<b>{band}</b>", "font": {"size": 14}},
        number={"suffix": " / 850", "font": {"size": 26}},
        gauge={
            "axis": {"range": [0, 850]},
            "bar": {"color": color, "thickness": 0.3},
            "steps": [{"range":[0,449],"color":"#fecaca"},{"range":[449,549],"color":"#fed7aa"},{"range":[549,649],"color":"#fef08a"},{"range":[649,749],"color":"#bbf7d0"},{"range":[749,850],"color":"#86efac"}]
        }
    ))
    fig_gauge.update_layout(height=260, margin=dict(l=10,r=10,t=30,b=10))
    st.plotly_chart(fig_gauge, use_container_width=True)
    st.caption(credit.get("band_description", ""))

with col_signals:
    signals = credit.get("signals", [])
    if signals:
        labels = [s["signal"].replace("_"," ").title() for s in signals]
        pcts   = [round(s["score"]/s["max"]*100) for s in signals]
        colors = ["#16a34a" if p>=70 else "#ca8a04" if p>=40 else "#dc2626" for p in pcts]
        fig_sig = go.Figure(go.Bar(x=pcts, y=labels, orientation="h", marker_color=colors, text=[f"{p}%" for p in pcts], textposition="outside"))
        fig_sig.update_layout(title="Signal Breakdown", xaxis=dict(range=[0,115]), template="plotly_white", height=260, margin=dict(l=5,r=20,t=30,b=10))
        st.plotly_chart(fig_sig, use_container_width=True)

with col_reco:
    st.caption("**Top Recommendations**")
    for rec in credit.get("recommendations", []):
        pct = round(rec[\'current_score\']/rec[\'max_score\']*100)
        color_emoji = "🟢" if pct >= 70 else "🟡" if pct >= 40 else "🔴"
        st.markdown(f"{color_emoji} **{rec[\'signal\'].replace(\'_\',\' \').title()}** ({pct}%)")
        st.caption(rec[\'advice\'])
        st.markdown("")

st.divider()


# ════════════════════════════════════════════════════════════════════
# SECTION 4 — AI ADVISOR
# ════════════════════════════════════════════════════════════════════
st.markdown(\'<div class="section-header">🤖 AI Business Advisor</div>\', unsafe_allow_html=True)

# Handle question from sidebar
question_to_use = user_question.strip() if (ask_btn and user_question.strip()) else None

# Generate advice (cached unless question changes)
@st.cache_data(ttl=300)
def get_advice_cached(merchant_id, question):
    eng = get_engine()
    a   = get_analytics(merchant_id, eng)
    c   = compute_credit_score(merchant_id, eng)
    return generate_ai_advice(merchant_id, a, c, question=question)

with st.spinner("Thinking..."):
    advice = get_advice_cached(merchant_id, question_to_use)

col_advice, col_meta = st.columns([3, 1])

with col_advice:
    if question_to_use:
        st.info(f"**Your question:** {question_to_use}")
    st.markdown(f\'<div class="advice-card"><div class="advice-text">💬 {advice["advice"]}</div></div>\', unsafe_allow_html=True)

with col_meta:
    st.caption("**Advisor Details**")
    st.caption(f"Model: {advice.get(\'model\', \'gpt-4o-mini\')}")
    st.caption(f"Tokens used: {advice.get(\'tokens_used\', 0)}")
    st.caption(f"Based on {analytics[\'transaction_count\']} transactions")
    st.markdown("")
    st.caption("**Quick questions**")
    quick_questions = [
        "How much loan can I get?",
        "Where is my money going?",
        "How do I improve my score?",
        "Is my business growing?",
    ]
    for q in quick_questions:
        if st.button(q, key=f"quick_{q[:10]}", use_container_width=True):
            st.cache_data.clear()
            with st.spinner("Thinking..."):
                quick_advice = generate_ai_advice(merchant_id, analytics, credit, question=q)
            st.info(f"**Q:** {q}")
            st.success(quick_advice["advice"])

st.divider()
st.caption(f"NombaIQ v1.0 · Merchant: {merchant_id} · Built on Nomba Webhooks")
'''

with open("dashboard/app.py", "w") as f:
    f.write(streamlit_app.strip())

print("dashboard/app.py written.")
print("   Run with: streamlit run dashboard/app.py")

## Write Streamlit Config

In [ ]:
os.makedirs(".streamlit", exist_ok=True)

streamlit_config = """[theme]
primaryColor = "#16a34a"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f8fafc"
textColor = "#0f172a"
font = "sans serif"

[server]
port = 8501
headless = true
enableCORS = false

[browser]
gatherUsageStats = false
"""

with open(".streamlit/config.toml", "w") as f:
    f.write(streamlit_config)

print(" .streamlit/config.toml written.")

## Preview All Charts in Notebook (Done Test)

In [ ]:
# ── THE DAY 6 DONE TEST ─────────────────────────────────────────────────
print("Running Day 6 Done Test — verifying all dashboard components...")
print()

# 1. Metrics banner data
checks = {
    "Section 1 — total_revenue_ngn":      analytics.get("total_revenue_ngn", 0) > 0,
    "Section 1 — net_cash_flow_ngn":      "net_cash_flow_ngn" in analytics,
    "Section 1 — wow_change_pct":         "wow_change_pct" in analytics,
    "Section 1 — avg_transaction_ngn":    analytics.get("avg_transaction_ngn", 0) > 0,
    "Section 2 — daily_revenue list":     len(analytics.get("daily_revenue", [])) > 0,
    "Section 2 — forecast_7day list":     len(analytics.get("forecast_7day", {}).get("forecast", [])) == 7,
    "Section 2 — category_breakdown":     len(analytics.get("category_breakdown", {})) > 0,
    "Section 3 — credit score 0–850":     0 <= credit.get("score", -1) <= 850,
    "Section 3 — 5 signals":              len(credit.get("signals", [])) == 5,
    "Section 3 — recommendations":        len(credit.get("recommendations", [])) > 0,
    "Section 4 — advice text present":    len(advice.get("advice", "")) > 20,
    "Section 4 — advice has naira (₦)":  "₦" in advice.get("advice", ""),
    "dashboard/app.py exists":            os.path.exists("dashboard/app.py"),
}

all_passed = all(checks.values())
for check, passed in checks.items():
    print(f"  {'Passed' if passed else 'Failed'} {check}")

print()
if all_passed:
    print(" DAY 6 DONE TEST PASSED — all 4 dashboard sections ready")
    print()
    print(" To launch the dashboard:")
    print("    streamlit run dashboard/app.py")
else:
    print(" Some checks failed — review cells above.")

print()
print(" Day 6 complete. Final step: Day 7 — Deploy + Demo.")